# Duomenu aibes aprasymas, transformacijos

#### Importuojame paketus, paruošiame grafikų stilių

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt
import contextily as cx

plt.style.use('bmh')
plt.rcParams['axes.facecolor'] = 'white'

## Duomenų importavimas

Visi failai pasiekiami OneDrive.

*Tolimesnio kodo paleidimas reikalauja daug RAM. Leisti kodą po "1 laboratorinio darbo kopija". Failas data.csv įkeltas į OneDrive.*

In [ ]:
df1 = pd.read_csv(
    'aisdk-2026-02-03.csv',
    engine='pyarrow'
)

df2 = pd.read_csv(
    'aisdk-2026-02-04.csv',
    engine='pyarrow'
)

df3 = pd.read_csv(
    'aisdk-2026-02-05.csv',
    engine='pyarrow'
)

In [ ]:
df = pd.concat([df1, df2, df3], axis=0)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 45522730 entries, 0 to 15750589
Data columns (total 26 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   # Timestamp                     str    
 1   Type of mobile                  str    
 2   MMSI                            int64  
 3   Latitude                        float64
 4   Longitude                       float64
 5   Navigational status             str    
 6   ROT                             float64
 7   SOG                             float64
 8   COG                             float64
 9   Heading                         float64
 10  IMO                             str    
 11  Callsign                        str    
 12  Name                            str    
 13  Ship type                       str    
 14  Cargo type                      str    
 15  Width                           float64
 16  Length                          float64
 17  Type of position fixing device  str    
 

## Filtruojam geografinį regioną, navigacinį statusą, `Type of mobile`

In [5]:
df_filtered = df[
    df['Latitude'].between(54, 56) &
    df['Longitude'].between(12, 17) &
    (df['Navigational status'] == 'Under way using engine') &
    (df['Type of mobile'] == 'Class A')
].copy(deep=True)

del df

## Ištiriam nelogiškas ar nereikalingas reikšmes

In [6]:
df_filtered = df_filtered[ ~(df_filtered['Ship type'] == 'Undefined') ]

> `Undefined` nereikia: mums reikia tik laivų, kuriu tipas yra žinomas, kad juos galėtume klasifikuoti, klasterizuoti ir t.t.

In [7]:
df_filtered.groupby('Ship type')['MMSI'].nunique()

Ship type
Cargo              330
Dredging             6
Fishing             15
HSC                  4
Law enforcement      7
Military             4
Other               16
Passenger           44
Pilot               12
Pleasure             1
SAR                 13
Spare 1              1
Tanker             119
Towing               2
Tug                 14
Name: MMSI, dtype: int64

Aišku, kad rinkinį dominuoja Tanker ir Cargo tipo laivai. Turime du realius variantus: (1) pasirenkame 3 klases (Cargo, Passenger ir Tanker) ir kiekvienos turime po 44 trajektorijas, (2) pasirenkame 2 klases (Cargo ir Tanker) ir turime po 119 stebinių. Prioritetą teiksime antram variantui.

In [8]:
df_filtered = df_filtered[df_filtered['Ship type'].isin(['Cargo', 'Tanker'])]

In [9]:
rng = np.random.default_rng(1)
selected_cargo_mmsi = rng.choice(df_filtered.loc[df_filtered['Ship type'] == 'Cargo', 'MMSI'].unique(), size=119, replace=False)

df_filtered = df_filtered[
    (df_filtered['Ship type'] != 'Cargo') |
    (df_filtered['MMSI'].isin(selected_cargo_mmsi))
].copy()

df_filtered.groupby('Ship type')['MMSI'].nunique()

Ship type
Cargo     119
Tanker    119
Name: MMSI, dtype: int64

In [10]:
df_filtered.to_csv('./data.csv')

# 1 laboratorinio darbo kopija

In [ ]:
df_filtered = pd.read_csv('./data.csv')

In [ ]:
df_filtered['Ship type'].value_counts()

In [ ]:
numerical = ['ROT', 'SOG', 'COG', 'Heading']
df_filtered[numerical].max()

In [ ]:
df_filtered[numerical].min()

In [ ]:
sns.boxplot(y='Ship type', x='SOG', hue='Ship type', data=df_filtered)

> Išskirtinai didelis SOG (100 mazgų ~ 185 kmh) yra paieškos ir gelbėjimo (*angl. Save and rescue, SAR*) laivo, kas, kontekste, nėra nelogiškas greitis

### Ištiriam praleistas reikšmes, skirtumus tarp stebėjimų

Čia mūsų siekis bus dvejopas:
1. Susitvarkyti su praleistomis reikšmėmis;
2. Išplėsti laiko eilutes į lygiai vienos minutės intervalus, t. y, padaryti taip, kad tarp vieno laivo AIS signalų visą laiką būtų vienos minutės skirtumas.

In [ ]:
df_filtered[numerical].isnull().sum()

### Heading praleistos reikšmės

In [ ]:
output = []

for ship_type, type_data in df_filtered.groupby('Ship type'):

    for col in numerical:
        output.append({
            'Laivo tipas': ship_type,
            'Kintamasis': col,
            'Praleistų reikšmių dalis, %': type_data[col].isnull().mean() * 100
        })

pd.DataFrame(output).pivot(columns=['Kintamasis'], index=['Laivo tipas'])

> Kadangi SAR tipo laivai turi virš 50 % praleistų reikšmių `Heading` ir `ROT` kintamuosiuose, šitą laivo tipą išimsime. Taip pat išimsime Tug laivo tipą, kadangi jo `ROT` požymis turi virš 50 % praleistų reikšmių

In [ ]:
df_filtered = df_filtered[
    ~( (df_filtered['Ship type'] == 'SAR') | 
       (df_filtered['Ship type'] == 'Tug') )
]

#### Praleistų reikšmių skaičius/dalis kiekvienam kintamajam

In [ ]:
df_filtered[numerical].isnull().agg(['sum', 'mean'])

> Toliau patikrinsime ar yra laivų, kurių kuriame nors AIS signalų rinkinyje yra 100 % praleistų reikšmių

In [ ]:
for mmsi, mmsi_data in df_filtered.groupby('MMSI'):

    if any( df_filtered[numerical].isnull().mean() >= 1 ):
        print(mmsi)

> Toliau reikia ištirti ar laiko skirtumas tarp stebėjimų neviršija 2 valandų (literatūroje taikyta riba)

In [ ]:
from datetime import timedelta

In [ ]:
df_filtered['# Timestamp'] = pd.to_datetime( df_filtered['# Timestamp'] )

In [ ]:
rm_mmsi = []
for mmsi, mmsi_data in df_filtered.groupby('MMSI'):

    mmsi_data = mmsi_data.sort_values('# Timestamp')

    time_diff = mmsi_data['# Timestamp'].diff(1)
    max_time_diff = time_diff.max()

    if max_time_diff >= timedelta(hours=2):
        rm_mmsi.append(mmsi)

print(f'Pašalinsime {len(rm_mmsi)} laivų')

In [ ]:
is_mmsi_removed = df_filtered['MMSI'].isin( set(rm_mmsi) )
df_filtered = df_filtered[ ~is_mmsi_removed ]

## isskirtys

In [ ]:

numerical_cols = ["ROT", "SOG", "COG", "Heading"]
for col in numerical_cols:
    df_col = df_filtered[col]
    
    Q1 = df_col.quantile(0.25)
    Q3 = df_col.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df_filtered[(df_filtered[col] < lower) | (df_filtered[col] > upper)]
    
    print(f"\n{col}")
    print("Lower:", round(lower, 2))
    print("Upper:", round(upper, 2))
    print("Min:", round(df_col.min(), 2))
    print("Max:", round(df_col.max(), 2))
    print("Outliers:", len(outliers))


In [ ]:
numerical_cols = ["ROT", "SOG"]

for ship_type, group in df_filtered.groupby("Ship type"):
    
    print(f"\n===== Ship type: {ship_type} =====")
    
    for col in numerical_cols:
        df_col = group[col].dropna()
        
        Q1 = df_col.quantile(0.25)
        Q3 = df_col.quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = group[(group[col] < lower) | (group[col] > upper)]
        
        print(f"\n{col}")
        print("Q1:", round(Q1, 2))
        print("Q3:", round(Q3, 2))
        print("Lower:", round(lower, 2))
        print("Upper:", round(upper, 2))
        print("Min:", round(df_col.min(), 2))
        print("Max:", round(df_col.max(), 2))
        print("Outliers:", len(outliers))


Taikant IQR metodą nustatyta, kad kintamasis ROT turi labai daug išskirčių. Tačiau tai paaiškinama tuo, kad didžioji dalis stebėjimų turi reikšmę 0 (laivas juda tiesiai), todėl kvartilių intervalas lygus nuliui ir visi nenuliniai stebėjimai klasifikuojami kaip išskirtys. Todėl pasirinkome išskirčių nešalinti.


# isimt ROT = 0 ir su boxplotu patikrint isskirtis

In [ ]:

numerical_cols = ["ROT", "SOG", "COG", "Heading"]
for col in numerical_cols:
    df_col = df_filtered[col]
    
    Q1 = df_col.quantile(0.25)
    Q3 = df_col.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df_filtered[(df_filtered[col] < lower) | (df_filtered[col] > upper)]
    
    print(f"\n{col}")
    print("Lower:", round(lower, 2))
    print("Upper:", round(upper, 2))
    print("Min:", round(df_col.min(), 2))
    print("Max:", round(df_col.max(), 2))
    print("Outliers:", len(outliers))


In [ ]:
df_isskirciu = df_filtered.copy()
df_isskirciu = df_isskirciu[df_isskirciu["ROT"] != 0]

In [ ]:

numerical_cols = ["ROT", "SOG", "COG", "Heading"]
for col in numerical_cols:
    df_col = df_isskirciu[col]
    
    Q1 = df_col.quantile(0.25)
    Q3 = df_col.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df_isskirciu[(df_isskirciu[col] < lower) | (df_isskirciu[col] > upper)]
    
    print(f"\n{col}")
    print("Lower:", round(lower, 2))
    print("Upper:", round(upper, 2))
    print("Min:", round(df_col.min(), 2))
    print("Max:", round(df_col.max(), 2))
    print("Outliers:", len(outliers))


In [ ]:
import matplotlib.pyplot as plt
import math

numerical_cols = ["ROT", "SOG"]

ship_types = df_isskirciu["Ship type"].dropna().unique()
n_ship_types = len(ship_types)
n_cols = len(numerical_cols)

# Grid dimensijos
rows = n_ship_types
cols = n_cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))

# Jei tik vienas ship type arba vienas column – sutvarkome indexing
if rows == 1:
    axes = [axes]
if cols == 1:
    axes = [[ax] for ax in axes]

for i, ship_type in enumerate(ship_types):
    group = df_isskirciu[df_isskirciu["Ship type"] == ship_type]
    
    for j, col in enumerate(numerical_cols):
        ax = axes[i][j]
        
        ax.boxplot(group[col].dropna())
        ax.set_title(f"{ship_type} - {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Value")

plt.tight_layout()
plt.show()


In [ ]:
filtered = df_isskirciu.loc[
    df_isskirciu["ROT"] > 150,
    # df_isskirciu["MMSI"] == 209325000,
    ["Ship type", "ROT", "Longitude", "Latitude", "# Timestamp", "MMSI"]
]

print(filtered)


In [ ]:
import folium
import pandas as pd

filtered = df_isskirciu.loc[
    df_isskirciu["MMSI"] == 209325000,
    ["Longitude", "Latitude", "# Timestamp"]
].copy()

filtered["# Timestamp"] = pd.to_datetime(filtered["# Timestamp"])
filtered = filtered.sort_values("# Timestamp")

center_lat = filtered["Latitude"].mean()
center_lon = filtered["Longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

coordinates = list(zip(filtered["Latitude"], filtered["Longitude"]))
folium.PolyLine(coordinates).add_to(m)

folium.Marker(
    coordinates[0],
    popup="Start",
    icon=folium.Icon(icon="play")
).add_to(m)

folium.Marker(
    coordinates[-1],
    popup="End",
    icon=folium.Icon(icon="stop")
).add_to(m)
m.save("trajectory_map.html")




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

ship_groups = [
    (ship_type, df_type)
    for ship_type, df_type in df_filtered.groupby("Ship type")
    #if len(df_type) >= 1000
]

n = len(ship_groups)

cols = 4
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(18, 4 * rows))
axes = axes.flatten()

palette = sns.color_palette("Set2", n)

for i, ((ship_type, df_type), color) in enumerate(zip(ship_groups, palette)):
    axes[i].hist(
        df_type["ROT"],
        bins=100,
        color=color,
        alpha=0.7
    )
    axes[i].set_title(ship_type)
    axes[i].set_xlabel("ROT")
    axes[i].set_ylabel("Frequency")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])


In [ ]:

plt.suptitle("ROT Distribution by Ship Type", fontsize=16)
plt.tight_layout()
plt.show()
plt.scatter(df_filtered["SOG"], df_filtered["ROT"], alpha=0.1)
plt.xlabel("SOG")
plt.ylabel("ROT")
plt.show()

### Praleistų reikšmių užpildymas: kubinė interpoliacija

In [ ]:
dfs = []
tarpai = []
praleistu_count = 0

for mmsi, mmsi_df in df_filtered.groupby('MMSI'):

    tarpai.extend( mmsi_df['# Timestamp'].diff(1).dropna().to_list() )
    
    # Atliekam transformacijq
    mmsi_df['Heading'] = mmsi_df['Heading']\
        .apply(lambda x: 360 - x if x > 180.0 else x)
    mmsi_df['COG'] = mmsi_df['COG']\
        .apply(lambda x: 360 - x if x > 180.0 else x)

    # Išplėčiame ir interpoliuojame
    output_df = mmsi_df.set_index('# Timestamp')\
        [['Latitude', 'Longitude', 'ROT', 'SOG', 'COG', 'Heading']]\
        .resample('60s')\
        .median()

    praleistu_count += output_df.isnull().sum()
    
    output_df = output_df\
        .interpolate('pchip')\
        .reset_index()

    # Ištaisome klaidas
    output_df['Heading'] = np.clip(output_df['Heading'], 0, 360)
    output_df['COG'] = np.clip(output_df['COG'], 0, 360)
    output_df['Latitude'] = np.clip(output_df['Latitude'], 54.0, 56.0)
    output_df['Longitude'] = np.clip(output_df['Longitude'], 12.0, 17.0)

    # Apskaičiuojame komponentes
    output_df['delta_lat'] = output_df['Latitude'].diff(1).abs()
    output_df['delta_lon'] = output_df['Longitude'].diff(1).abs()

    # Priskiriame MMSI ir laivo tipo reikšmes, nes šiuo metu jų nėra output_df
    output_df['MMSI'] = mmsi
    output_df['Ship type'] = mmsi_df['Ship type'].values[0]

    # Surenkame
    dfs.append(output_df)

data = pd.concat(dfs, ignore_index=True).dropna()

In [ ]:
tarpai = np.array(tarpai)
tarpai = tarpai[ tarpai != 0 ]

np.quantile(tarpai, [0.05, 0.95])

> Tarpai tarp AIS stebėjimų yra 0 - 20 sekundžių

In [ ]:
praleistu_count / data.shape[0]

> Interpoliuota nuo 5 iki 15 % reikšmių priklausomai nuo požymio

In [ ]:
print('Stebėjimų skaičius', data.shape[0])
print('Laivų skaičius', data['MMSI'].nunique())

## Rinkinio peržiūra

In [ ]:
data = data.sort_values(['MMSI', '# Timestamp'])

In [ ]:
data.head()

In [ ]:
gdf = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.Longitude, data.Latitude), crs="EPSG:4326"
)

##### Pavyzdinė tanklaivio judėjimo trajektorija

fig, ax = plt.subplots(1, 1, figsize=(15, 15))

gdf[gdf.MMSI == 538007403].plot(ax=ax, markersize=12, color='red')

cx.add_basemap(ax, crs=gdf.crs, source=cx.providers.OpenStreetMap.Mapnik, zoom=9)

plt.title('Laivas ID 538007403, 2026-02-05 00:05-13:04')

# plt.savefig('plots/1lab/path1.png', format='png', dpi=300, bbox_inches="tight")

##### Pavyzdinė krovininio laivo judėjimo trajektorija

fig, ax = plt.subplots(1, 1, figsize=(15, 15))

gdf[gdf.MMSI == 636022047].plot(ax=ax, markersize=12, color='blue')

cx.add_basemap(ax, crs=gdf.crs, source=cx.providers.OpenStreetMap.Mapnik, zoom=9)

plt.title('Laivas ID 636022047, 2026-02-05 06:22-18:04')

# plt.savefig('plots/1lab/path2.png', format='png', dpi=600, bbox_inches="tight")

### Reikšmių normavimas

In [ ]:

import pandas as pd

numeric_cols = data[numerical]

Q1 = numeric_cols.quantile(0.25)
Q3 = numeric_cols.quantile(0.75)
min_vals = numeric_cols.min()
max_vals = numeric_cols.max()

desc_stats = pd.DataFrame({
    'Minimumas': min_vals,
    '$Q_1$': Q1,
    'Vidurkis': numeric_cols.mean(),
    '$Q_3$': Q3,
    '$IQR$': Q3 - Q1,
    'Maksimumas': max_vals,
    'Standartinis nuokrypis': numeric_cols.std()
})

pd.options.display.float_format = '{:.3f}'.format
print(desc_stats.to_latex(float_format='{:.3f}'.format))


In [ ]:
data['ROT'] = np.arcsinh(data['ROT'])

# kampų transformacija
heading_rad = np.deg2rad(data['Heading'])
cog_rad = np.deg2rad(data['COG'])

data['Heading_sin'] = np.sin(heading_rad)
data['Heading_cos'] = np.cos(heading_rad)

data['COG_sin'] = np.sin(cog_rad)
data['COG_cos'] = np.cos(cog_rad)

# pašaliname originalius kampus
# data = data.drop(columns=['Heading', 'COG'])

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
standscaler = StandardScaler()
minmaxscaler = MinMaxScaler()
numerical = [
    'Latitude',
    'Longitude',
    'ROT',
    'SOG',
    'Heading_sin',
    'Heading_cos',
    'COG_sin',
    'COG_cos',
    'delta_lon',
    'delta_lat'
]
standscaler.fit(data[numerical])
minmaxscaler.fit(data[numerical])

In [ ]:
standdata = data.copy()
standdata[numerical] = standscaler.transform(data[numerical])

In [ ]:
standdata[numerical].describe()

In [ ]:
sns.displot(standdata[standdata.ROT != standdata.ROT.median()]['ROT'], height=6, aspect=2)

In [ ]:
sns.boxplot(standdata[standdata.ROT != standdata.ROT.median()]['ROT'])

In [ ]:
data[numerical] = minmaxscaler.transform(data[numerical])

In [ ]:
data[numerical].describe()

In [ ]:
sns.displot(data[data.ROT != data.ROT.median()]['ROT'], height=6, aspect=2)

In [ ]:
sns.displot(data[data.ROT != data.ROT.median()]['ROT'], height=6, aspect=2)

In [ ]:
sns.boxplot(data[data.ROT != data.ROT.median()]['ROT'])

### Koreliacijos

In [ ]:
corr = data[numerical]\
    .select_dtypes(include=['float64'])\
    .corr(method='spearman')

In [ ]:
corr

# paziureti koreliacijas pagal laivu tipus

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math

ship_types = data["Ship type"].dropna().unique()

n = len(ship_types)
cols = 3  # kiek heatmap vienoje eilėje
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 6 * rows))
axes = axes.flatten()

for i, ship_type in enumerate(ship_types):
    
    subset = data[data["Ship type"] == ship_type]
    
    columns = numerical.copy()
    if subset['ROT'].nunique() == 1:
        columns.remove('ROT')

    corr = subset[columns].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool))

    sns.heatmap(
        corr,
        mask=mask,
        cmap="Reds",
        annot=True,
        fmt=".2f",
        ax=axes[i],
        cbar=False,
        annot_kws={"fontsize": 18},
        vmin=0,
        vmax=1
    )
    
    axes[i].set_title(f"{ship_type}", size=24)
    axes[i].tick_params(axis='both', labelsize=16)

# Paslepiame nenaudojamus subplot'us jei jų liko
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
# plt.savefig('plots/1lab/corr-matrices.png', dpi=750)
plt.show()


In [ ]:

ship_types = ['Tanker', 'Passenger']

n = len(ship_types)
cols = 2
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 8 * rows))
axes = axes.flatten()

for i, ship_type in enumerate(ship_types):
    
    subset = data[data["Ship type"] == ship_type].copy(deep=True)[numerical]
    subset = subset.rename(columns={'delta_lat': r'$\delta_{lat}$', 'delta_lon': r'$\delta_{lon}$'})
    
    columns = numerical.copy()
    if subset['ROT'].nunique() == 1:
        columns.remove('ROT')

    corr = subset.corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool))

    sns.heatmap(
        corr,
        mask=mask,
        cmap="Reds",
        annot=True,
        fmt=".2f",
        ax=axes[i],
        cbar=False,
        annot_kws={"fontsize": 18},
        vmin=0,
        vmax=1
    )
    
    axes[i].set_title(f"{ship_type}", size=24)
    axes[i].tick_params(axis='both', labelsize=20)

# Paslepiame nenaudojamus subplot'us jei jų liko
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plots/1lab/corr-matrices.png', dpi=750)
plt.show()


## Aprasomoji statistika, palyginimoji analizė

### Aprasomoji statistika

In [ ]:
import pandas as pd

numeric_cols = data[numerical]

Q1 = numeric_cols.quantile(0.25)
Q3 = numeric_cols.quantile(0.75)
min_vals = numeric_cols.min()
max_vals = numeric_cols.max()

desc_stats = pd.DataFrame({
    'min': min_vals,
    'Q1': Q1,
    'median': numeric_cols.median(),
    'mean': numeric_cols.mean(),
    'Q3': Q3,
    'IQR': Q3 - Q1,
    'max': max_vals,
    'range': max_vals - min_vals,
    'variance': numeric_cols.var(),
    'std_dev': numeric_cols.std()
})

pd.options.display.float_format = '{:.3f}'.format
print(desc_stats)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

ship_counts = (
    data.groupby("Ship type")["MMSI"]
        .nunique()
        .sort_values(ascending=True)
)

palette = [
    "#02010A", "#140152", "#5C374C", "#662C91", 
    "#2473B9", "#81A4CD", "#E3DFFF", "#E56399",
    "#880D1E", "#DD2D4A", "#DD7230", "#EDD382"
]
colors = palette[:len(ship_counts)]

total_ships = ship_counts.sum()
top_val = ship_counts.values[-1]
top_pct = (top_val / total_ships) * 100

fig, ax = plt.subplots(figsize=(13, 7))
y_pos = np.arange(len(ship_counts))

min_visual_width = 1 
visual_lengths = [max(v, min_visual_width) for v in ship_counts.values]

ax.barh(
    y_pos, 
    visual_lengths, 
    color=colors, 
    height=0.85     
)

for y, real_value, vis_length in zip(y_pos, ship_counts.values, visual_lengths):
    label_text = f"{real_value:,}".replace(',', ' ')
    text_x_pos = vis_length + 2 

    ax.text(
        text_x_pos,
        y,
        label_text,
        va='center',
        fontsize=20,
        color='#444444'
    )

ax.annotate(
    f"{top_pct:.1f}".replace(".", ",") + "% visų laivų",
    xy=(
        top_val,
        len(ship_counts)-1 - 0.42 
    ),
    xytext=(top_val - 35, len(ship_counts)-4.5),
    ha='center',
    arrowprops=dict(
        arrowstyle="->",
        color="#444444",
        connectionstyle="angle3,angleA=0,angleB=90",
        lw=1.5
    ),
    fontsize=25,
    color="#444444",
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.8)
)

ax.xaxis.set_major_locator(ticker.MultipleLocator(20))

ax.set_yticks(y_pos)
ax.set_yticklabels(ship_counts.index, fontsize=15, fontweight='medium')

ax.set_xlim(0, 170)
ax.set_ylim(-1.5, len(ship_counts))

ax.set_xlabel("Laivų skaičius", fontsize=25, color='black') 

ax.grid(axis='y', visible=False)
ax.grid(axis='x', color='gray', alpha=0.1, linewidth=1)
ax.set_axisbelow(True)

ax.spines[['top','right','left','bottom']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', color='gray', labelsize=20)

plt.tight_layout()
plt.show()

### Palyginamoji analizė

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_box_by_shiptype(df, variables):
    
    dfp = df.dropna(subset=["Ship type"]).copy()
    dfp["Ship type"] = dfp["Ship type"].astype(str).str.strip()
    
    variables = [
        col for col in variables 
        if col in dfp.columns and pd.api.types.is_numeric_dtype(dfp[col])
    ]
    
    sns.set_theme(style="ticks", context="talk")

    ship_order = [
        "Cargo",
        "Tanker",
        "Pilot",
        "Passenger",
        "Fishing",
        "Other",
        "Dredging",
        "HSC",
        "Towing",
        "Spare 1",
        "Law enforcement",
        "Military"
    ]

    color_map = {
        "Cargo": "#EDD382",
        "Tanker": "#DD7230",
        "Pilot": "#DD2D4A",
        "Passenger": "#880D1E",
        "Fishing": "#E56399",
        "Other": "#E3DFFF",
        "Dredging": "#81A4CD",
        "HSC": "#2473B9",
        "Towing": "#662C91",
        "Spare 1": "#5C374C",
        "Law enforcement": "#140152",
        "Military": "#989899B5"
    }  

    ship_order = [s for s in ship_order if s in dfp["Ship type"].unique()]

    for var in variables:
        
        plt.figure(figsize=(16, 8))
        
        valid_ship_types = [
            s for s in ship_order
            if s in dfp["Ship type"].unique()
            and dfp.loc[dfp["Ship type"] == s, var].notna().any()
        ]
        
        sns.boxplot(
        data=dfp,
        x="Ship type",
        y=var,
        order=valid_ship_types,
        palette=color_map,
        width=0.5,
        linewidth=1.2,
        showfliers=False
    )

        ax = plt.gca()
        ax.yaxis.grid(True, which='major', color="#D6CECEE7", linewidth=0.8)
        ax.xaxis.grid(False)

        plt.xticks(rotation=90, ha="right", fontsize=20)
        # plt.title(f"{var} pasiskirstymas pagal laivo tipą", fontsize=25, weight="bold")
        plt.xlabel("Laivo tipas", fontsize=25)
        plt.ylabel(var, fontsize=25)

        sns.despine()
        plt.tight_layout()
        plt.show()


plot_box_by_shiptype(df_isskirciu, selected_vars)

In [ ]:
print( data[numerical].describe().drop(['count', 'mean', 'std']).T.to_latex() )

## Sekų formavimas

In [ ]:
for mmsi, mmsi_data in data.groupby('MMSI'):

    sample = mmsi_data[[
        'Latitude',
        'Longitude',
        'ROT',
        'SOG',
        'Heading_sin',
        'Heading_cos',
        'COG_sin',
        'COG_cos',
        'delta_lat',
        'delta_lon'
    ]]
    ship_type = mmsi_data['Ship type'].values[0]

    sequences = np.empty( (sample.shape[0], 25, sample.shape[1]) )

    for ix, col in enumerate( sample.columns ):
        for lag in range( 25 ):
            sequences[:, lag, ix] = sample[col].shift(-1 * lag).T

    with open(f'sequences/seq-{mmsi}-{ship_type}.npy', 'wb+') as f:
        np.save(f, sequences[:-25:25, :, :])